# NB 03 - Agents & Orchestrator

The 5-agent claims pipeline (Intake, Validation, Fraud, Assessment, Resolution) + orchestrator. Uses Cortex COMPLETE, SENTIMENT, EMBED, SEARCH, and Guardrails.

## 05_agents_orchestrator.SQL

Insurance Claims Agentic AI - Snowflake GCC Hackathon
ALL 5 AGENTS + ORCHESTRATOR (Full Cortex enabled)

> Cortex functions used across agents:  
> Intake      : CORTEX.COMPLETE (entity extraction, LOB classification)  
> Validation  : SQL policy lookup + multi-LOB cross-reference  
> Fraud       : CORTEX.COMPLETE (narrative) + CORTEX.SENTIMENT  
> + CORTEX.EMBED_TEXT_768 / VECTOR_COSINE_SIMILARITY (embed sim)  
> + CORTEX.SEARCH_PREVIEW (RAG fraud pattern retrieval)  
> Assessment  : CORTEX.SEARCH_PREVIEW (RAG guidelines) + CORTEX.COMPLETE  
> Resolution  : CORTEX.COMPLETE with guardrails=TRUE  
>   
> Note: All Python SPs use string concatenation (not nested $$)  
> to avoid the nested dollar-quote delimiter compilation error.

In [ ]:
%%sql -r dataframe_1
USE ROLE INSURANCE_APP_ROLE;
USE DATABASE INSURANCE_DB;
USE WAREHOUSE AGENT_WH;

## AGENT 1: INTAKE AGENT (Cortex LLM - Entity Extraction)

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE PROCEDURE PROCESSED.SP_AGENT_INTAKE(claim_id VARCHAR)
RETURNS VARIANT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'run_intake_agent'
EXECUTE AS CALLER
AS
$$
import json
import time
from snowflake.snowpark import Session

def run_intake_agent(session: Session, claim_id: str) -> dict:
    start_time = time.time()
    
    claim_row = session.sql(
        "SELECT claim_id, policy_id, customer_id, claim_text, "
        "incident_date, claimed_amount, lob_type, incident_location "
        "FROM RAW.CLAIMS_LANDING WHERE claim_id = '" + claim_id + "'"
    ).collect()
    
    if not claim_row:
        return {"status": "ERROR", "error": "Claim " + claim_id + " not found"}
    
    claim = claim_row[0]
    claim_text = str(claim['CLAIM_TEXT'] or '')
    
    system_prompt = (
        "You are an expert insurance claims intake processor. "
        "Extract structured information from the claim description. "
        "Return ONLY valid JSON with keys: "
        "claimant_name, policy_number, incident_date, incident_location, "
        "lob_classification (AUTO|PROPERTY|WORKERS_COMP), lob_confidence (0.0-1.0), "
        "claimed_amount, damage_description, "
        "urgency_level (LOW|MEDIUM|HIGH|CRITICAL), "
        "multi_lob_indicator (true/false), "
        "estimated_severity (MINOR|MODERATE|SEVERE|CATASTROPHIC)"
    )
    
    user_prompt = (
        "Extract entities from this claim:\\n"
        "CLAIM: " + claim_text[:2000] + "\\n"
        "Policy: " + str(claim['POLICY_ID'] or '') + "\\n"
        "Customer: " + str(claim['CUSTOMER_ID'] or '') + "\\n"
        "Date: " + str(claim['INCIDENT_DATE'] or '') + "\\n"
        "Amount: " + str(claim['CLAIMED_AMOUNT'] or '') + "\\n"
        "Return JSON only."
    )
    
    sys_esc = system_prompt.replace("'", "''")
    usr_esc = user_prompt.replace("'", "''")
    
    sql = (
        "SELECT SNOWFLAKE.CORTEX.COMPLETE("
        "'mistral-large2', "
        "ARRAY_CONSTRUCT("
        "OBJECT_CONSTRUCT('role','system','content','" + sys_esc + "'),"
        "OBJECT_CONSTRUCT('role','user','content','" + usr_esc + "')"
        "),"
        "OBJECT_CONSTRUCT('temperature',0.1,'max_tokens',2000)"
        ") AS R"
    )
    
    try:
        result = session.sql(sql).collect()
        raw = result[0]['R']
        resp = json.loads(raw)
        content = resp.get('choices',[{}])[0].get('messages','')
        if not content:
            content = resp.get('choices',[{}])[0].get('message',{}).get('content','')
        usage = resp.get('usage',{})
        tokens_in = usage.get('prompt_tokens',0)
        tokens_out = usage.get('completion_tokens',0)
    except Exception as e:
        content = ''
        tokens_in = 0
        tokens_out = 0
    
    try:
        clean = content.strip()
        if clean.startswith('```'):
            clean = clean.split('\\n',1)[1] if '\\n' in clean else clean[3:]
            clean = clean.rsplit('```',1)[0]
        extracted = json.loads(clean)
    except:
        extracted = {"lob_classification": claim['LOB_TYPE'] or "UNKNOWN", "lob_confidence": 0.5, "parse_error": True}
    
    latency_ms = int((time.time() - start_time) * 1000)
    
    output = {
        "status": "SUCCESS",
        "agent": "INTAKE",
        "claim_id": claim_id,
        "extracted_data": extracted,
        "lob_classification": extracted.get('lob_classification', claim['LOB_TYPE']),
        "confidence_score": extracted.get('lob_confidence', 0.0),
        "urgency_level": extracted.get('urgency_level', 'MEDIUM'),
        "multi_lob_indicator": extracted.get('multi_lob_indicator', False),
        "latency_ms": latency_ms,
        "tokens_input": tokens_in,
        "tokens_output": tokens_out
    }
    
    out_esc = json.dumps(output).replace("'", "''")
    
    session.sql(
        "MERGE INTO PROCESSED.CLAIM_STATE cs "
        "USING (SELECT '" + claim_id + "' AS claim_id) src ON cs.claim_id = src.claim_id "
        "WHEN MATCHED THEN UPDATE SET current_state='INTAKE_COMPLETE', "
        "intake_output=PARSE_JSON('" + out_esc + "') "
        "WHEN NOT MATCHED THEN INSERT(claim_id,current_state,intake_output,started_at) "
        "VALUES('" + claim_id + "','INTAKE_COMPLETE',PARSE_JSON('" + out_esc + "'),CURRENT_TIMESTAMP())"
    ).collect()
    
    session.sql(
        "INSERT INTO RESULTS.AUDIT_LOG(flow_type,reference_id,agent_name,step_number,"
        "cortex_module_used,model_used,tokens_input,tokens_output,latency_ms,status) "
        "SELECT 'CLAIMS','" + claim_id + "','INTAKE_AGENT',1,"
        "'CORTEX_COMPLETE','mistral-large2'," + str(tokens_in) + "," + str(tokens_out) + ","
        + str(latency_ms) + ",'SUCCESS'"
    ).collect()
    
    return output
$$;

## AGENT 2: VALIDATION AGENT (Policy Verification)

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE PROCEDURE PROCESSED.SP_AGENT_VALIDATION(claim_id VARCHAR)
RETURNS VARIANT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'run_validation_agent'
EXECUTE AS CALLER
AS
$$
import json
import time
from snowflake.snowpark import Session

def run_validation_agent(session: Session, claim_id: str) -> dict:
    start_time = time.time()
    
    claim = session.sql(
        "SELECT cl.claim_id, cl.policy_id, cl.customer_id, cl.lob_type, "
        "cl.claimed_amount, cl.incident_date "
        "FROM RAW.CLAIMS_LANDING cl WHERE cl.claim_id = '" + claim_id + "'"
    ).collect()
    
    if not claim:
        return {"status": "ERROR", "error": "Claim not found"}
    
    c = claim[0]
    policy_id = str(c['POLICY_ID'] or '')
    customer_id = str(c['CUSTOMER_ID'] or '')
    claimed_amount = float(c['CLAIMED_AMOUNT'] or 0)
    incident_date = str(c['INCIDENT_DATE'] or '')
    lob_type = str(c['LOB_TYPE'] or '')
    
    # Policy lookup
    policy = session.sql(
        "SELECT policy_id, customer_id, lob_type, policy_status, "
        "start_date, end_date, coverage_limit, deductible, "
        "underwriting_tier, geographic_zone "
        "FROM RAW.POLICIES WHERE policy_id = '" + policy_id + "'"
    ).collect()
    
    if not policy:
        result = {"eligible": False, "reason": "POLICY_NOT_FOUND"}
    else:
        p = policy[0]
        checks = []
        eligible = True
        
        # Check 1: Status
        if p['POLICY_STATUS'] != 'ACTIVE':
            eligible = False
            checks.append({"check":"STATUS","passed":False,"detail":"Policy is " + str(p['POLICY_STATUS'])})
        else:
            checks.append({"check":"STATUS","passed":True,"detail":"ACTIVE"})
        
        # Check 2: Dates
        if incident_date < str(p['START_DATE']) or incident_date > str(p['END_DATE']):
            eligible = False
            checks.append({"check":"DATES","passed":False,"detail":"Incident outside coverage period"})
        else:
            checks.append({"check":"DATES","passed":True,"detail":"Within coverage period"})
        
        # Check 3: LOB match
        if lob_type and p['LOB_TYPE'] != lob_type:
            eligible = False
            checks.append({"check":"LOB","passed":False,"detail":"Mismatch: " + lob_type + " vs " + str(p['LOB_TYPE'])})
        else:
            checks.append({"check":"LOB","passed":True,"detail":"LOB matches"})
        
        # Check 4: Coverage limit
        coverage_limit = float(p['COVERAGE_LIMIT'] or 0)
        deductible = float(p['DEDUCTIBLE'] or 0)
        if claimed_amount > coverage_limit:
            checks.append({"check":"LIMIT","passed":False,"detail":"Exceeds limit"})
        else:
            checks.append({"check":"LIMIT","passed":True,"detail":"Within limit"})
        
        # Multi-LOB check
        multi = session.sql(
            "SELECT COUNT(DISTINCT lob_type) AS cnt FROM RAW.POLICIES "
            "WHERE customer_id='" + customer_id + "' AND policy_status='ACTIVE'"
        ).collect()
        multi_lob_count = int(multi[0]['CNT']) if multi else 0
        
        # Concurrent claims
        concurrent = session.sql(
            "SELECT COUNT(*) AS cnt FROM RAW.CLAIMS_LANDING "
            "WHERE customer_id='" + customer_id + "' "
            "AND incident_date='" + incident_date + "' "
            "AND claim_id != '" + claim_id + "'"
        ).collect()
        concurrent_count = int(concurrent[0]['CNT']) if concurrent else 0
        
        result = {
            "eligible": eligible,
            "coverage_limit": coverage_limit,
            "deductible": deductible,
            "max_payable": min(claimed_amount, coverage_limit) - deductible if eligible else 0,
            "checks": checks,
            "multi_lob_count": multi_lob_count,
            "concurrent_claims": concurrent_count,
            "is_multi_claim_event": concurrent_count > 0
        }
    
    latency_ms = int((time.time() - start_time) * 1000)
    
    output = {
        "status": "SUCCESS",
        "agent": "VALIDATION",
        "claim_id": claim_id,
        "validation_result": result,
        "eligible": result.get('eligible', False),
        "coverage_limit": result.get('coverage_limit', 0),
        "deductible": result.get('deductible', 0),
        "max_payable": result.get('max_payable', 0),
        "latency_ms": latency_ms
    }
    
    out_esc = json.dumps(output).replace("'", "''")
    
    session.sql(
        "UPDATE PROCESSED.CLAIM_STATE SET current_state='VALIDATION_COMPLETE', "
        "validation_output=PARSE_JSON('" + out_esc + "') "
        "WHERE claim_id='" + claim_id + "'"
    ).collect()
    
    session.sql(
        "INSERT INTO RESULTS.AUDIT_LOG(flow_type,reference_id,agent_name,step_number,"
        "cortex_module_used,latency_ms,status) "
        "SELECT 'CLAIMS','" + claim_id + "','VALIDATION_AGENT',2,"
        "'SQL_POLICY_LOOKUP'," + str(latency_ms) + ",'SUCCESS'"
    ).collect()
    
    return output
$$;

## AGENT 3: FRAUD DETECTION AGENT (Search + Embed + LLM)

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE PROCEDURE PROCESSED.SP_AGENT_FRAUD(claim_id VARCHAR)
RETURNS VARIANT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'run_fraud_agent'
EXECUTE AS CALLER
AS
$$
import json
import time
from snowflake.snowpark import Session

def run_fraud_agent(session: Session, claim_id: str) -> dict:
    start_time = time.time()
    
    claim = session.sql(
        "SELECT cl.claim_id, cl.policy_id, cl.customer_id, cl.lob_type, "
        "cl.claim_text, cl.claimed_amount, cl.incident_date "
        "FROM RAW.CLAIMS_LANDING cl WHERE cl.claim_id='" + claim_id + "'"
    ).collect()
    
    if not claim:
        return {"status": "ERROR", "error": "Claim not found"}
    
    c = claim[0]
    claim_text = str(c['CLAIM_TEXT'] or '')
    customer_id = str(c['CUSTOMER_ID'] or '')
    policy_id = str(c['POLICY_ID'] or '')
    lob_type = str(c['LOB_TYPE'] or '')
    claimed_amount = float(c['CLAIMED_AMOUNT'] or 0)
    incident_date = str(c['INCIDENT_DATE'] or '')
    
    # SUB-CHECK 1: Duplicate detection (SQL)
    dups = session.sql(
        "SELECT COUNT(*) AS cnt FROM RAW.CLAIMS_LANDING "
        "WHERE customer_id='" + customer_id + "' "
        "AND policy_id='" + policy_id + "' "
        "AND claim_id != '" + claim_id + "' "
        "AND ABS(DATEDIFF(day, incident_date, '" + incident_date + "')) <= 90"
    ).collect()
    dup_count = int(dups[0]['CNT']) if dups else 0
    dup_score = min(dup_count * 0.3, 0.9)
    
    # SUB-CHECK 2: Customer tenure check
    tenure = session.sql(
        "SELECT DATEDIFF(month, onboarding_date, CURRENT_DATE()) AS months "
        "FROM RAW.CUSTOMERS WHERE customer_id='" + customer_id + "'"
    ).collect()
    tenure_months = int(tenure[0]['MONTHS']) if tenure else 0
    tenure_score = 0.6 if tenure_months < 6 else (0.3 if tenure_months < 12 else 0.1)
    
    # SUB-CHECK 3: Embedding similarity to fraud patterns (Cortex Embed + Vector)
    # Uses VECTORS.COMPUTE_FRAUD_SIMILARITY() which calls EMBED_TEXT_768 + VECTOR_COSINE_SIMILARITY
    embed_score = 0.0
    try:
        embed = session.sql(
            "SELECT INSURANCE_DB.VECTORS.COMPUTE_FRAUD_SIMILARITY('" 
            + claim_text[:2000].replace("'", "''") + "') AS score"
        ).collect()
        embed_score = float(embed[0]['SCORE'] or 0) if embed else 0
    except:
        embed_score = 0.0
    
    # SUB-CHECK 3B: Cortex Search RAG - retrieve similar historical fraud patterns
    rag_score = 0.0
    matched_patterns = []
    try:
        search_query = claim_text[:400].replace('"', ' ').replace("\\n", " ").replace("'", "")
        rag = session.sql(
            "SELECT PARSE_JSON(SNOWFLAKE.CORTEX.SEARCH_PREVIEW("
            "'INSURANCE_DB.VECTORS.FRAUD_PATTERNS_SEARCH_SERVICE', "
            "'{\"query\": \"" + search_query + "\", "
            "\"columns\": [\"pattern_type\", \"severity\"], "
            "\"limit\": 3}')) AS R"
        ).collect()
        if rag and rag[0]['R']:
            rag_results = json.loads(rag[0]['R']) if isinstance(rag[0]['R'], str) else rag[0]['R']
            sev_map = {"CRITICAL": 0.9, "HIGH": 0.7, "MEDIUM": 0.5, "LOW": 0.3}
            for i, res in enumerate(rag_results.get('results', [])):
                relevance = 1.0 - (i * 0.25)
                sev = res.get('severity', 'LOW')
                rag_score = max(rag_score, relevance * sev_map.get(sev, 0.3))
                matched_patterns.append({"pattern": res.get('pattern_type'), "severity": sev})
    except:
        rag_score = 0.0
    
    # SUB-CHECK 4: Sentiment analysis (Cortex Sentiment)
    try:
        sent = session.sql(
            "SELECT SNOWFLAKE.CORTEX.SENTIMENT('" 
            + claim_text[:2000].replace("'", "''") + "') AS score"
        ).collect()
        sentiment = float(sent[0]['SCORE'] or 0) if sent else 0
    except:
        sentiment = 0.0
    
    # SUB-CHECK 5: Narrative analysis via LLM (Cortex Complete)
    narrative_prompt = (
        "Analyze this insurance claim for fraud signals. "
        "Score 0.0-1.0: vagueness, inconsistency, emotional_manipulation, urgency_pressure. "
        "Return JSON: {overall_risk: 0.0-1.0, red_flags: [], green_flags: []}\\n\\n"
        "CLAIM: " + claim_text[:1500]
    ).replace("'", "''")
    
    try:
        narr = session.sql(
            "SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-large2','" + narrative_prompt + "') AS R"
        ).collect()
        narr_raw = narr[0]['R'] if narr else '{}'
        narr_json = json.loads(narr_raw)
        narr_content = narr_json.get('choices',[{}])[0].get('messages','') or \
                       narr_json.get('choices',[{}])[0].get('message',{}).get('content','{}')
        try:
            narr_data = json.loads(narr_content.strip().strip('`').strip('json').strip())
        except:
            narr_data = {}
        narrative_score = float(narr_data.get('overall_risk', 0.3))
        red_flags = narr_data.get('red_flags', [])
        green_flags = narr_data.get('green_flags', [])
    except:
        narrative_score = 0.3
        red_flags = []
        green_flags = []
    
    # COMPOSITE SCORE (weighted - now includes Cortex Search RAG signal)
    # duplicate(15%) + tenure(10%) + embed(20%) + rag(15%) + narrative(25%) + sentiment(15%)
    composite = (
        dup_score * 0.15 +
        tenure_score * 0.10 +
        embed_score * 0.20 +
        rag_score * 0.15 +
        narrative_score * 0.25 +
        (0.15 if sentiment < -0.3 else 0.0)  # negative sentiment adds risk
    )
    composite = min(max(composite, 0.0), 1.0)
    
    if composite > 0.75:
        risk_level = "HIGH"
        action = "REFER_TO_SIU"
    elif composite > 0.45:
        risk_level = "MEDIUM"
        action = "FLAG_FOR_REVIEW"
    else:
        risk_level = "LOW"
        action = "PASS"
    
    latency_ms = int((time.time() - start_time) * 1000)
    
    output = {
        "status": "SUCCESS",
        "agent": "FRAUD_DETECTION",
        "claim_id": claim_id,
        "composite_fraud_score": round(composite, 4),
        "risk_level": risk_level,
        "recommended_action": action,
        "sub_scores": {
            "duplicate": dup_score,
            "tenure_risk": tenure_score,
            "embedding_similarity": embed_score,
            "rag_pattern_match": rag_score,
            "narrative_risk": narrative_score,
            "sentiment": sentiment
        },
        "matched_fraud_patterns": matched_patterns,
        "red_flags": red_flags,
        "green_flags": green_flags,
        "latency_ms": latency_ms
    }
    
    out_esc = json.dumps(output).replace("'", "''")
    
    session.sql(
        "UPDATE PROCESSED.CLAIM_STATE SET current_state='FRAUD_COMPLETE', "
        "fraud_output=PARSE_JSON('" + out_esc + "') "
        "WHERE claim_id='" + claim_id + "'"
    ).collect()
    
    session.sql(
        "INSERT INTO RESULTS.AUDIT_LOG(flow_type,reference_id,agent_name,step_number,"
        "cortex_module_used,latency_ms,status) "
        "SELECT 'CLAIMS','" + claim_id + "','FRAUD_AGENT',3,"
        "'COMPLETE+EMBED+SEARCH+SENTIMENT'," + str(latency_ms) + ",'SUCCESS'"
    ).collect()
    
    return output
$$;

## AGENT 4: ASSESSMENT AGENT (Settlement Calculation)

In [ ]:
%%sql -r dataframe_5
CREATE OR REPLACE PROCEDURE PROCESSED.SP_AGENT_ASSESSMENT(claim_id VARCHAR)
RETURNS VARIANT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'run_assessment_agent'
EXECUTE AS CALLER
AS
$$
import json
import time
from snowflake.snowpark import Session

def run_assessment_agent(session: Session, claim_id: str) -> dict:
    start_time = time.time()
    
    # Get claim + upstream outputs
    data = session.sql(
        "SELECT cl.claimed_amount, cl.lob_type, cl.claim_text, "
        "cs.validation_output, cs.fraud_output "
        "FROM RAW.CLAIMS_LANDING cl "
        "JOIN PROCESSED.CLAIM_STATE cs ON cl.claim_id = cs.claim_id "
        "WHERE cl.claim_id='" + claim_id + "'"
    ).collect()
    
    if not data:
        return {"status": "ERROR", "error": "Claim not found in state"}
    
    d = data[0]
    claimed_amount = float(d['CLAIMED_AMOUNT'] or 0)
    lob_type = str(d['LOB_TYPE'] or '')
    
    # Parse upstream
    validation = json.loads(d['VALIDATION_OUTPUT']) if d['VALIDATION_OUTPUT'] else {}
    fraud = json.loads(d['FRAUD_OUTPUT']) if d['FRAUD_OUTPUT'] else {}
    
    coverage_limit = float(validation.get('coverage_limit', 0))
    deductible = float(validation.get('deductible', 0))
    fraud_score = float(fraud.get('composite_fraud_score', 0))
    
    # Settlement formula
    base = max(min(claimed_amount, coverage_limit) - deductible, 0)
    
    if fraud_score > 0.75:
        risk_factor = 0.0
        risk_reason = "HIGH fraud - settlement withheld"
    elif fraud_score > 0.45:
        risk_factor = 0.7
        risk_reason = "MEDIUM fraud - 30% reduction"
    else:
        risk_factor = 1.0
        risk_reason = "LOW fraud - no adjustment"
    
    settlement = base * risk_factor
    
    # RAG: Retrieve relevant underwriting/settlement guidelines (Cortex Search)
    guideline_context = ""
    try:
        gq = (lob_type + " claim settlement guidelines coverage").replace("'", "")
        guide = session.sql(
            "SELECT PARSE_JSON(SNOWFLAKE.CORTEX.SEARCH_PREVIEW("
            "'INSURANCE_DB.VECTORS.UNDERWRITING_GUIDELINES_SEARCH_SERVICE', "
            "'{\"query\": \"" + gq + "\", "
            "\"columns\": [\"section_name\", \"content\"], "
            "\"limit\": 2}')) AS R"
        ).collect()
        if guide and guide[0]['R']:
            gdata = json.loads(guide[0]['R']) if isinstance(guide[0]['R'], str) else guide[0]['R']
            parts = []
            for res in gdata.get('results', []):
                parts.append(str(res.get('section_name','')) + ": " + str(res.get('content',''))[:300])
            guideline_context = " | ".join(parts)
    except:
        guideline_context = ""
    
    # Generate rationale via LLM (grounded with RAG guidelines)
    rationale_prompt = (
        "You are an insurance claims assessor. Summarize this settlement in 2-3 sentences, "
        "citing relevant guidelines. "
        "Claim: " + str(claimed_amount) + " INR, "
        "Limit: " + str(coverage_limit) + ", "
        "Deductible: " + str(deductible) + ", "
        "Fraud score: " + str(round(fraud_score, 2)) + ", "
        "Settlement: " + str(round(settlement, 0)) + " INR. "
        "Risk: " + risk_reason + ". "
        "GUIDELINES: " + guideline_context[:800]
    ).replace("'", "''")
    
    try:
        rat = session.sql(
            "SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-large2','" + rationale_prompt + "') AS R"
        ).collect()
        raw_rat = rat[0]['R'] if rat else ''
        rat_json = json.loads(raw_rat)
        rationale = rat_json.get('choices',[{}])[0].get('messages','') or \
                    rat_json.get('choices',[{}])[0].get('message',{}).get('content', raw_rat)
    except:
        rationale = "Settlement of INR " + str(int(settlement)) + " calculated. " + risk_reason
    
    latency_ms = int((time.time() - start_time) * 1000)
    
    output = {
        "status": "SUCCESS",
        "agent": "ASSESSMENT",
        "claim_id": claim_id,
        "settlement_amount": round(settlement, 2),
        "calculation": {
            "claimed": claimed_amount,
            "coverage_limit": coverage_limit,
            "deductible": deductible,
            "base_settlement": base,
            "fraud_score": fraud_score,
            "risk_factor": risk_factor,
            "final_settlement": settlement
        },
        "assessment_rationale": rationale,
        "guidelines_used": guideline_context[:500],
        "latency_ms": latency_ms
    }
    
    out_esc = json.dumps(output).replace("'", "''")
    
    session.sql(
        "UPDATE PROCESSED.CLAIM_STATE SET current_state='ASSESSMENT_COMPLETE', "
        "assessment_output=PARSE_JSON('" + out_esc + "') "
        "WHERE claim_id='" + claim_id + "'"
    ).collect()
    
    session.sql(
        "INSERT INTO RESULTS.AUDIT_LOG(flow_type,reference_id,agent_name,step_number,"
        "cortex_module_used,latency_ms,status) "
        "SELECT 'CLAIMS','" + claim_id + "','ASSESSMENT_AGENT',4,"
        "'COMPLETE+SEARCH'," + str(latency_ms) + ",'SUCCESS'"
    ).collect()
    
    return output
$$;

## AGENT 5: RESOLUTION AGENT (Final Decision + Guardrails)

In [ ]:
%%sql -r dataframe_6
CREATE OR REPLACE PROCEDURE PROCESSED.SP_AGENT_RESOLUTION(claim_id VARCHAR)
RETURNS VARIANT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'run_resolution_agent'
EXECUTE AS CALLER
AS
$$
import json
import time
from snowflake.snowpark import Session

def run_resolution_agent(session: Session, claim_id: str) -> dict:
    start_time = time.time()
    
    # Get all upstream data
    data = session.sql(
        "SELECT cs.intake_output, cs.validation_output, cs.fraud_output, cs.assessment_output, "
        "cl.claimed_amount, cl.lob_type, cl.customer_id, cl.policy_id, "
        "c.customer_segment, c.lifetime_value_score "
        "FROM PROCESSED.CLAIM_STATE cs "
        "JOIN RAW.CLAIMS_LANDING cl ON cs.claim_id = cl.claim_id "
        "JOIN RAW.CUSTOMERS c ON cl.customer_id = c.customer_id "
        "WHERE cs.claim_id='" + claim_id + "'"
    ).collect()
    
    if not data:
        return {"status": "ERROR", "error": "Claim state not found"}
    
    d = data[0]
    customer_id = str(d['CUSTOMER_ID'] or '')
    policy_id = str(d['POLICY_ID'] or '')
    lob_type = str(d['LOB_TYPE'] or '')
    claimed_amount = float(d['CLAIMED_AMOUNT'] or 0)
    segment = str(d['CUSTOMER_SEGMENT'] or 'STANDARD')
    ltv = float(d['LIFETIME_VALUE_SCORE'] or 0)
    
    validation = json.loads(d['VALIDATION_OUTPUT']) if d['VALIDATION_OUTPUT'] else {}
    fraud = json.loads(d['FRAUD_OUTPUT']) if d['FRAUD_OUTPUT'] else {}
    assessment = json.loads(d['ASSESSMENT_OUTPUT']) if d['ASSESSMENT_OUTPUT'] else {}
    
    eligible = validation.get('eligible', True)
    fraud_score = float(fraud.get('composite_fraud_score', 0))
    fraud_risk = fraud.get('risk_level', 'LOW')
    settlement = float(assessment.get('settlement_amount', 0))
    
    # Decision logic
    if not eligible:
        decision = 'DENIED'
        reason = 'Policy validation failed'
        settlement_final = 0
    elif fraud_score > 0.75:
        decision = 'REFERRED'
        reason = 'High fraud risk (score: ' + str(round(fraud_score, 2)) + ')'
        settlement_final = 0
    elif fraud_score > 0.45 and settlement > 0:
        decision = 'PARTIALLY_APPROVED'
        reason = 'Medium fraud risk - reduced settlement'
        settlement_final = settlement
    elif settlement > 0:
        decision = 'APPROVED'
        reason = 'All checks passed'
        settlement_final = settlement
    else:
        decision = 'DENIED'
        reason = 'Zero settlement calculated'
        settlement_final = 0
    
    confidence = 0.95 if fraud_score < 0.3 else (0.7 if fraud_score < 0.6 else 0.5)
    
    # Generate customer-facing summary via LLM with Cortex Guardrails enabled
    # Guardrails=true filters unsafe/inappropriate content from the LLM response
    summary_prompt = (
        "Write a professional 2-sentence claim resolution summary for the policyholder. "
        "Do not include internal fraud scores or PII. "
        "Decision: " + decision + ". "
        "Settlement: INR " + str(int(settlement_final)) + " of " + str(int(claimed_amount)) + " claimed. "
        "Reason: " + reason
    ).replace("'", "''")
    
    try:
        # ARRAY_CONSTRUCT messages form + options with guardrails
        summ = session.sql(
            "SELECT SNOWFLAKE.CORTEX.COMPLETE("
            "'mistral-large2', "
            "ARRAY_CONSTRUCT(OBJECT_CONSTRUCT('role','user','content','" + summary_prompt + "')), "
            "OBJECT_CONSTRUCT('temperature', 0.3, 'max_tokens', 300, 'guardrails', TRUE)"
            ") AS R"
        ).collect()
        raw_summ = summ[0]['R'] if summ else ''
        summ_json = json.loads(raw_summ)
        summary = summ_json.get('choices',[{}])[0].get('messages','') or \
                  summ_json.get('choices',[{}])[0].get('message',{}).get('content', reason)
    except:
        # Fallback without guardrails option if not supported
        try:
            summ = session.sql(
                "SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-large2','" + summary_prompt + "') AS R"
            ).collect()
            raw_summ = summ[0]['R'] if summ else ''
            summ_json = json.loads(raw_summ)
            summary = summ_json.get('choices',[{}])[0].get('messages','') or \
                      summ_json.get('choices',[{}])[0].get('message',{}).get('content', reason)
        except:
            summary = "Claim " + claim_id + " " + decision + ". " + reason
    
    # NBA for HNW customers
    nba = None
    if segment == 'HNW' and ltv > 0.7:
        concurrent = session.sql(
            "SELECT COUNT(*) AS cnt FROM RAW.CLAIMS_LANDING "
            "WHERE customer_id='" + customer_id + "' AND incident_date = "
            "(SELECT incident_date FROM RAW.CLAIMS_LANDING WHERE claim_id='" + claim_id + "')"
        ).collect()
        if concurrent and int(concurrent[0]['CNT']) > 1:
            nba = {"action": "EXPEDITE", "reason": "HNW multi-LOB disaster - VIP treatment"}
    
    latency_ms = int((time.time() - start_time) * 1000)
    
    output = {
        "status": "SUCCESS",
        "agent": "RESOLUTION",
        "claim_id": claim_id,
        "decision": decision,
        "decision_reason": reason,
        "settlement_amount": round(settlement_final, 2),
        "confidence_score": confidence,
        "resolution_summary": summary,
        "next_best_action": nba,
        "latency_ms": latency_ms
    }
    
    out_esc = json.dumps(output).replace("'", "''")
    summary_esc = summary.replace("'", "''")
    
    # Write to RESOLUTIONS
    session.sql(
        "INSERT INTO RESULTS.RESOLUTIONS"
        "(claim_id,customer_id,policy_id,lob_type,decision,"
        "settlement_amount,claimed_amount,fraud_risk_level,fraud_score,"
        "reasoning_summary,confidence_score,processing_time_ms) "
        "SELECT '" + claim_id + "','" + customer_id + "','" + policy_id + "','" + lob_type + "',"
        "'" + decision + "'," + str(settlement_final) + "," + str(claimed_amount) + ","
        "'" + fraud_risk + "'," + str(fraud_score) + ","
        "'" + summary_esc + "'," + str(confidence) + "," + str(latency_ms)
    ).collect()
    
    # Update state
    session.sql(
        "UPDATE PROCESSED.CLAIM_STATE SET current_state='COMPLETED', "
        "resolution_output=PARSE_JSON('" + out_esc + "'), "
        "completed_at=CURRENT_TIMESTAMP() "
        "WHERE claim_id='" + claim_id + "'"
    ).collect()
    
    # Update claim status
    session.sql(
        "UPDATE RAW.CLAIMS_LANDING SET claim_status='" + decision + "' "
        "WHERE claim_id='" + claim_id + "'"
    ).collect()
    
    # Audit
    session.sql(
        "INSERT INTO RESULTS.AUDIT_LOG(flow_type,reference_id,agent_name,step_number,"
        "cortex_module_used,latency_ms,status) "
        "SELECT 'CLAIMS','" + claim_id + "','RESOLUTION_AGENT',5,"
        "'COMPLETE+GUARDRAILS'," + str(latency_ms) + ",'SUCCESS'"
    ).collect()
    
    return output
$$;

## ORCHESTRATOR: Simple SQL (chains all 5 agents)

In [ ]:
%%sql -r dataframe_7
CREATE OR REPLACE PROCEDURE PROCESSED.SP_PROCESS_CLAIM(claim_id VARCHAR)
RETURNS VARCHAR
LANGUAGE SQL
EXECUTE AS CALLER
AS
$$
DECLARE
    decision VARCHAR;
BEGIN
    CALL PROCESSED.SP_AGENT_INTAKE(:claim_id);
    CALL PROCESSED.SP_AGENT_VALIDATION(:claim_id);
    CALL PROCESSED.SP_AGENT_FRAUD(:claim_id);
    CALL PROCESSED.SP_AGENT_ASSESSMENT(:claim_id);
    CALL PROCESSED.SP_AGENT_RESOLUTION(:claim_id);
    
    decision := (
        SELECT decision FROM RESULTS.RESOLUTIONS 
        WHERE claim_id = :claim_id 
        ORDER BY decided_at DESC LIMIT 1
    );
    
    RETURN 'Claim ' || :claim_id || ' processed. Decision: ' || COALESCE(:decision, 'UNKNOWN');
END;
$$;

## BATCH ORCHESTRATOR

In [ ]:
%%sql -r dataframe_8
CREATE OR REPLACE PROCEDURE PROCESSED.SP_PROCESS_ALL_CLAIMS()
RETURNS VARCHAR
LANGUAGE SQL
EXECUTE AS CALLER
AS
$$
DECLARE
    cnt INT DEFAULT 0;
    cur_claim VARCHAR;
BEGIN
    FOR rec IN (
        SELECT claim_id FROM RAW.CLAIMS_LANDING 
        WHERE claim_status = 'SUBMITTED' 
        ORDER BY submission_date ASC LIMIT 20
    ) DO
        cur_claim := rec.claim_id;
        CALL PROCESSED.SP_PROCESS_CLAIM(:cur_claim);
        cnt := cnt + 1;
    END FOR;
    
    RETURN 'Batch complete. Processed ' || :cnt::VARCHAR || ' claims.';
END;
$$;

## HELPER SQL FUNCTIONS (no procedure issues)

In [ ]:
%%sql -r dataframe_9
CREATE OR REPLACE FUNCTION PROCESSED.CLASSIFY_LOB(claim_text VARCHAR)
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
    SELECT SNOWFLAKE.CORTEX.COMPLETE(
        'mistral-large2',
        CONCAT('Classify into one LOB. Reply with ONLY: AUTO or PROPERTY or WORKERS_COMP\n\nClaim: ', LEFT(claim_text, 1000))
    )
$$;

CREATE OR REPLACE FUNCTION PROCESSED.CLAIM_SENTIMENT(claim_text VARCHAR)
RETURNS FLOAT
LANGUAGE SQL
AS
$$
    SELECT SNOWFLAKE.CORTEX.SENTIMENT(claim_text)
$$;

CREATE OR REPLACE FUNCTION PROCESSED.SUMMARIZE_CLAIM(claim_text VARCHAR)
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
    SELECT SNOWFLAKE.CORTEX.SUMMARIZE(claim_text)
$$;

## VIEWS (Pipeline Status)

In [ ]:
%%sql -r dataframe_10
CREATE OR REPLACE VIEW PROCESSED.V_PIPELINE_STATUS AS
SELECT
    cs.claim_id,
    cl.customer_id,
    cl.lob_type,
    cl.claimed_amount,
    cs.current_state,
    cs.started_at,
    cs.completed_at,
    DATEDIFF(second, cs.started_at, COALESCE(cs.completed_at, CURRENT_TIMESTAMP())) AS elapsed_seconds,
    cs.fraud_output:composite_fraud_score::FLOAT AS fraud_score,
    cs.fraud_output:risk_level::VARCHAR AS fraud_risk,
    cs.resolution_output:decision::VARCHAR AS decision,
    cs.resolution_output:settlement_amount::FLOAT AS settlement,
    c.customer_segment,
    c.lifetime_value_score
FROM PROCESSED.CLAIM_STATE cs
JOIN RAW.CLAIMS_LANDING cl ON cs.claim_id = cl.claim_id
JOIN RAW.CUSTOMERS c ON cl.customer_id = c.customer_id
ORDER BY cs.started_at DESC;

## TEST: Run the full pipeline

In [ ]:
%%sql -r dataframe_12
CALL PROCESSED.SP_PROCESS_CLAIM('CLM-001-AUTO');  
CALL PROCESSED.SP_PROCESS_CLAIM('CLM-005-AUTO');  
--CALL PROCESSED.SP_PROCESS_ALL_CLAIMS();  
SELECT * FROM PROCESSED.V_PIPELINE_STATUS;  
SELECT * FROM RESULTS.RESOLUTIONS;

In [ ]:
%%sql -r dataframe_11
